# Phase 3 Decision-Making Log

## Purpose

This notebook records the **methodological reasoning behind Phase 3 decisions** for the E-Commerce Pricing Analytics project.

It is a decision/audit log, not the implementation notebook. The purpose is to document **why** each modelling choice was made so that the later modelling notebook can implement the already-agreed methodology consistently.

No model-performance results are recorded here because the final regression models have not yet been trained.


## 1. Source and Scope

Phase 3 uses the validated processed dataset produced in Phase 2:

`data/processed/books_processed_data.csv`

The project specification states that Phase 2 produced 1,000 observations and 6 variables:

`title`, `price`, `availability`, `rating`, `category`, and `upc`.

The Phase 2 specification also states that `product_description` was excluded during preprocessing and that categorical encoding was intentionally left for Phase 3.


## 2. Data Readiness Observations

The processed dataset contains **1,000 observations and 6 variables**.

For `price`, the observed statistics are:

| Statistic | Value |
|---|---:|
| Mean | 35.0703 |
| Median | 35.9800 |
| Minimum | 10.00 |
| Maximum | 59.99 |
| Standard deviation | 14.4467 |
| Skewness | -0.0375 |

The IQR-based price-outlier check gives:

* Q1 = 22.1075
* Q3 = 47.4575
* IQR = 25.3500
* Lower boundary = -15.9175
* Upper boundary = 85.4825
* Observations outside the boundaries = **0**

The dataset contains **50 category levels**.

These are observations from the processed data, not model-performance results.


## 3. Decision: Predict `price`

### Decision

`price` is the continuous target variable.

### Reasoning

The project objective is price prediction, so `price` is the response variable and must not be included as an input feature.

The task is therefore a supervised regression problem:

**Features → predicted price**


## 4. Decision: Use `availability`, `rating`, and `category`

### Decision

The modelling features are:

* `availability`
* `rating`
* `category`

### Reasoning

These are validated product attributes retained by Phase 2 and have interpretable relationships to the prediction task.

Keeping the feature set structured avoids turning Phase 3 into an unnecessary text/NLP modelling problem.

The variables are handled according to their data types and modelling roles rather than being included simply because they exist in the dataset.


## 5. Decision: Exclude `title`

### Decision

`title` is excluded from the modelling features.

### Reasoning

`title` is free text. The agreed Phase 3 project scope is structured-feature regression rather than NLP feature extraction.

Using TF-IDF or other text representations would introduce a substantially different modelling workflow and a high-dimensional feature space for a dataset of only 1,000 observations.

Excluding it keeps the modelling problem focused, interpretable, and aligned with the validated structured variables from Phase 2.


## 6. Decision: Exclude `upc`

### Decision

`upc` is excluded from modelling.

### Reasoning

`upc` identifies a product; it is not a meaningful numerical measurement or an explanatory product attribute.

Treating an identifier as a numeric predictor could create artificial numerical relationships. One-hot encoding an identifier would instead create a near-unique category for each product and would not provide useful generalisable information.

The Phase 2 specification explicitly characterises UPC values as product identifiers rather than numerical measurements.


## 7. Decision: Treat `rating` as Numeric

### Decision

`rating` is treated as a numeric ordinal variable on its validated 1–5 scale.

### Reasoning

The rating has a natural ordered structure from 1 to 5. Treating it numerically preserves this ordering without creating five separate dummy variables.

It will be included with the numeric predictors and scaled for models that use the common preprocessing pipeline.


## 8. Decision: Keep All 50 Categories

### Decision

All 50 validated source categories are retained. Rare categories are not manually merged into an `Other` category.

### Reasoning

Phase 2 validated and retained all source categories. Merging categories would introduce an additional subjective preprocessing rule and would modify information that was already validated during cleaning.

Potentially sparse category levels can instead be handled consistently by the encoding pipeline.

Any limitations associated with rare categories can be discussed after model evaluation rather than removing them in advance without evidence that removal is necessary.


## 9. Decision: One-Hot Encode `category`

### Decision

`category` is transformed using `OneHotEncoder`.

The agreed settings are:

* `drop="first"`
* `handle_unknown="ignore"`

### Reasoning

Category labels do not have a meaningful numerical ordering, so assigning arbitrary integers would create false ordinal relationships.

One-hot encoding represents category membership without imposing numerical order.

`drop="first"` avoids redundant dummy variables for the linear regression model.

`handle_unknown="ignore"` makes the fitted preprocessing pipeline robust when a category appears at prediction time that was not observed during fitting.


## 10. Decision: Standardise Numeric Predictors

### Decision

`availability` and `rating` are passed through `StandardScaler`.

### Reasoning

Scaling puts numeric predictors onto a comparable scale for models whose optimisation or distance calculations can be affected by feature magnitude, particularly SVR and linear modelling.

Random Forest does not require scaling for its tree splits, but using a model-specific pipeline keeps preprocessing explicit and prevents inconsistent manual transformations.

The scaler must be fitted only on training data through the modelling pipeline to prevent leakage.


## 11. Decision: 80/20 Train-Test Split

### Decision

Use an 80% training set and a 20% test set, with `random_state=42`.

### Reasoning

The dataset contains 1,000 observations, so an 80/20 split provides a substantial training sample while reserving an independent test sample for final evaluation.

The fixed random state makes the split reproducible.

Most importantly, the test set is not used for tuning or model selection.


## 12. Decision: 5-Fold K-Fold Cross-Validation

### Decision

Use 5-fold K-Fold cross-validation within the training data.

### Reasoning

A single train/validation split would make tuning more dependent on one particular partition.

Five-fold cross-validation repeatedly trains and validates on different portions of the training data, providing a more stable estimate of training-data generalisation for model comparison and hyperparameter tuning.

The final test set remains untouched throughout this process.


## 13. Decision: DummyRegressor Baseline

### Decision

Include a `DummyRegressor(strategy="mean")` baseline.

### Reasoning

A baseline establishes how well a very simple strategy performs without learning relationships between predictors and price.

For this project, the mean-price baseline provides a direct reference point. Candidate machine-learning models should demonstrate predictive value beyond this baseline.


## 14. Decision: Three Regression Models

### Decision

Use:

1. Multiple Linear Regression
2. Random Forest Regression
3. Support Vector Regression

### Reasoning

The three models represent meaningfully different modelling assumptions:

* Linear Regression provides an interpretable linear reference.
* Random Forest provides a nonlinear tree-based approach.
* SVR provides an alternative regression approach capable of nonlinear modelling depending on its kernel.

This provides methodological variety without expanding the project into an unnecessarily large model zoo.


## 15. Decision: Do Not Tune Linear Regression

### Decision

Multiple Linear Regression is fitted without GridSearchCV hyperparameter tuning.

### Reasoning

Standard multiple linear regression does not have the same model-complexity hyperparameters that are being explored for Random Forest and SVR.

It therefore serves as a transparent, untuned reference model. The model comparison remains meaningful because the tuned nonlinear models are compared against both this linear reference and the DummyRegressor baseline.


## 16. Decision: Use GridSearchCV for Random Forest and SVR

### Decision

Random Forest and SVR will be tuned using `GridSearchCV` with 5-fold cross-validation.

### Reasoning

Both models expose hyperparameters that can materially affect model complexity and predictive behaviour.

Grid search provides a systematic and reproducible way to evaluate predefined hyperparameter combinations using the same cross-validation procedure.

The search is performed only on the training data. The test set must not influence hyperparameter selection.


## 17. Decision: Use RMSE as the GridSearchCV Scoring Criterion

### Decision

RMSE is the primary scoring criterion for GridSearchCV.

In scikit-learn, RMSE is represented using the corresponding negative-error scoring convention because GridSearchCV maximises scoring values.

### Reasoning

RMSE is expressed in the original price units and penalises larger prediction errors more strongly than MAE.

Using one clearly defined primary criterion makes hyperparameter selection consistent across the tuned models.

The configuration with the lowest mean cross-validated RMSE on the training data will be preferred.


## 18. Decision: Report RMSE + MAE + R²

### Decision

The final evaluation reports:

* RMSE — primary metric
* MAE — complementary error metric
* R² — complementary explanatory metric

### Why RMSE?

RMSE measures prediction error in the same units as price and gives greater weight to larger errors. This makes it appropriate as the primary criterion when large pricing mistakes should receive stronger penalties.

### Why add MAE?

MAE is less sensitive to individual large errors than RMSE and represents the average absolute prediction error in the original price units.

Reporting both RMSE and MAE distinguishes two useful aspects of error behaviour:

* RMSE shows how strongly larger errors affect performance.
* MAE gives a more directly typical absolute-error interpretation.

Therefore, MAE is not replacing RMSE; it complements it.

### Why R²?

R² provides a different perspective by describing the proportion of variation in price explained by the model.

R² should therefore not be interpreted as another error-unit measure. It complements RMSE and MAE.


## 19. Decision: No Log Transformation of `price`

### Decision

Use the original `price` scale.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/processed/books_processed_data.csv')

In [2]:
df['price'].describe()

count    1000.00000
mean       35.07035
std        14.44669
min        10.00000
25%        22.10750
50%        35.98000
75%        47.45750
max        59.99000
Name: price, dtype: float64

### Evidence

The processed data show:

* mean ≈ 35.07
* median ≈ 35.98
* skewness ≈ -0.037
* minimum = 10.00
* maximum = 59.99

The mean and median are close and the skewness is very close to zero.

### Reasoning

There is no strong distributional evidence in the processed target that would justify adding a log transformation solely to correct substantial right-skewness.

Keeping the original scale also makes RMSE and MAE directly interpretable as price-unit errors.

A transformation can be reconsidered only if later diagnostic evidence from the fitted models provides a specific modelling reason.


## 20. Decision: No Outlier Removal or Winsorisation

### Decision

Do not remove or winsorise price observations.

### Evidence

The IQR-based assessment found **zero price observations outside the IQR outlier boundaries**.

### Reasoning

There is no evidence from this check that the price variable contains extreme observations requiring automatic removal.

Removing observations without a demonstrated data-quality problem would unnecessarily alter the validated Phase 2 dataset.


## 21. Decision: Test Set Is Untouched

### Decision

The 20% test set is used only for the final evaluation after model selection.

### Reasoning

If the test set were repeatedly used to choose models or hyperparameters, it would no longer provide an independent estimate of out-of-sample performance.

The agreed workflow is:

**Training data → 5-fold CV / GridSearchCV → model selection → final test evaluation**

This separation is a central leakage-control decision.


## 22. Decision: Select the Final Model Using Mean CV RMSE

### Decision

The final candidate model is selected using the **lowest mean cross-validated RMSE on the training data**.

### Reasoning

Model selection must occur without using the test set.

RMSE is already the project's primary modelling criterion, so using mean CV RMSE for selection keeps the tuning and selection logic consistent.

After selection, the untouched test set is evaluated once using RMSE, MAE, and R².


## 23. Decision: Reproducibility

### Decision

Use `random_state=42` where applicable and use 5-fold K-Fold cross-validation.

### Reasoning

Fixed randomisation makes the train-test split and stochastic model components reproducible.

The goal is that another run of the project follows the same methodological procedure and can reproduce the modelling workflow and evaluation setup.


## 24. Agreed Phase 3 Workflow

The final agreed workflow is:

**Phase 2 processed data**
→ feature selection  
→ 80/20 train-test split  
→ training-only 5-fold K-Fold CV  
→ preprocessing pipelines  
→ DummyRegressor baseline  
→ Multiple Linear Regression  
→ GridSearchCV for Random Forest  
→ GridSearchCV for SVR  
→ model selection using lowest mean CV RMSE  
→ untouched test-set evaluation  
→ RMSE + MAE + R²  
→ save the final fitted preprocessing-and-model pipeline.

This notebook records the reasoning behind these choices. The actual implementation and model results belong in the subsequent Phase 3 modelling notebook.
